# Vision BiLRP — image similarity with VGG-16

Paper-faithful reimplementation of BiLRP (Eberle, Büttner, Kräutli, Müller, Valleriani,
Montavon, *Building and Interpreting Deep Similarity Models*, IEEE TPAMI 2022):

* **Similarity model**: dot product on the *full spatial* activations of VGG-16 layer 31
  (the end of `vgg.features` — 512 maps at 4×4 for 128px inputs), as in the paper §5.
* **LRP rules**: the paper's depth-dependent γ schedule — γ = 0.5 / 0.25 / 0.1 / 0.0 for
  layers 2–10 / 11–17 / 18–24 / 25–31 — plus the z^B (zbox) rule at the input convolution.
  The depth grouping is expressed with autoLRP **analyzers**: a fact is a config key.
* **Efficiency**: the paper's random-projection layer (100 dims), LRP-propagated like any layer.
* **Conservation**: BiLRP satisfies Σ R_pair = ⟨φ(a),φ(b)⟩ (paper Prop. 1, exact for zero-bias
  nets); we print the ratio — VGG's conv biases absorb a fraction, which is expected.

In [ ]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
from PIL import Image

import autolrp
from autolrp import LRPConfig, BASE
import _common  # noqa  (applies the showcase rcParams)

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# 128px crop is the BiLRP paper convention — smaller field of view
# concentrates relevance on the subject.
preprocess = transforms.Compose([
    transforms.Resize(128), transforms.CenterCrop(128),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

def denorm(t):
    d = t[0].clone()
    for c, (m, s) in enumerate(zip(MEAN, STD)):
        d[c] = d[c] * s + m
    return d.clamp(0, 1).permute(1, 2, 0).numpy()

cat0 = preprocess(Image.open('../../../data/cat0.jpg').convert('RGB')).unsqueeze(0)
cat1 = preprocess(Image.open('../../../data/cat1.jpg').convert('RGB')).unsqueeze(0)

In [ ]:
vgg = torchvision.models.vgg16(weights='DEFAULT').eval()

class Encoder(nn.Module):
    """phi(x) = full spatial activations of VGG-16 layer 31 (end of
    `features`), flattened — the paper's similarity model uses the raw
    dot product on these activations (no global pooling, no cosine)."""
    def __init__(self, features):
        super().__init__()
        self.features = features
    def forward(self, x):
        return self.features(x).flatten(1)

encoder = Encoder(vgg.features).eval()

with torch.no_grad():
    ea, eb = encoder(cat0), encoder(cat1)
    dot = float((ea * eb).sum())
print(f'phi dim = {ea.shape[-1]}  (512 x 4 x 4 spatial)')
print(f'dot-product similarity <phi(a), phi(b)> = {dot:.1f}')

## The paper's rule composite via analyzers

The γ schedule needs *different rules at different depths* — grad_fn names can't express
that, so we register **analyzers** that tag each `ConvolutionBackward` node by its depth
group (walk order is output→input). The facts are config keys; the input conv is
routed to z^B by the built-in `input_conv` analyzer fact
(bounds = normalized-pixel range, stated explicitly in the rule dict).

We then sum-pool each side's pixel relevance to an 8×8 grid (the paper's coarse-graining
`R_II' = Σ_{i∈I}Σ_{i'∈I'} R_ii'` — sum pooling preserves conservation) and take the
outer product per projected dimension.

In [ ]:
from autolrp.backward import analysis as A

def _convs(nodes):
    return [n for n in nodes if 'ConvolutionBackward' in n.name()]

# BFS-from-output plan order: convs appear top-of-network first. Paper
# depth groups (13 convs; the deepest = the input conv, position 12, is
# NOT tagged here — it gets the z^B rule via the built-in 'input_conv'
# analyzer fact instead):
_GROUPS = {'vgg_gamma_00':  (0, 2),    # layers 25-31: gamma = 0.0 (LRP-0)
           'vgg_gamma_01':  (2, 5),    # layers 18-24: gamma = 0.1
           'vgg_gamma_025': (5, 8),    # layers 11-17: gamma = 0.25
           'vgg_gamma_05':  (8, 12)}   # layers  2-10: gamma = 0.5
for _name, (_lo, _hi) in _GROUPS.items():
    @A.register_analyzer(_name)
    def _tag(nodes, _lo=_lo, _hi=_hi):
        return {n: True for n in _convs(nodes)[_lo:_hi]}

# z^B input bounds in normalized-pixel space: (0-mean)/std .. (1-mean)/std
ZB_LOW, ZB_HIGH = (0 - 0.485) / 0.229, (1 - 0.406) / 0.225

paper_cfg = LRPConfig(
    rule={**BASE,                          # epsilon on every family not addressed below
          'vgg_gamma_00':  'epsilon',      # paper gamma=0.0 == LRP-0 == epsilon
          'vgg_gamma_01':  ('gamma', {'gamma': 0.1}),
          'vgg_gamma_025': ('gamma', {'gamma': 0.25}),
          'vgg_gamma_05':  ('gamma', {'gamma': 0.5}),
          'input_conv':    ('zbox', {'low': ZB_LOW, 'high': ZB_HIGH})},  # layer 1: z^B
)

POOL = 8

def reduce_to_8x8(r):
    # SUM pooling (paper's conservation-preserving coarse-graining):
    # sum channels, then sum each (H/8 x W/8) cell.
    s = r.sum(dim=1, keepdim=True)
    cell = s.shape[-1] // POOL
    return (F.avg_pool2d(s, cell) * cell**2).flatten(start_dim=1)

torch.manual_seed(42)
R_pair, sim_proj = autolrp.bilrp(
    encoder, cat0, cat1,
    n_dims=100,                       # the paper's random-projection width
    reduce_each=reduce_to_8x8,
    config=paper_cfg,
    return_similarity=True,
)
print(f'R_pair shape: {tuple(R_pair.shape)}')
print(f'projected similarity (decomposition target): {sim_proj:.1f}')
print(f'sum R_pair: {float(R_pair.sum()):.1f}   '
      f'conservation ratio: {float(R_pair.sum())/sim_proj:.3f} '
      f'(paper Prop. 1; <1 expected — VGG biases absorb relevance)')

In [ ]:
# Bipartite visualization: side-by-side images with the top-K most
# relevant patch pairs drawn as lines. Red = positive contribution
# to similarity, blue = negative. Line width ∝ |R[i, j]|.
import matplotlib.lines as mlines
from matplotlib.gridspec import GridSpec

TOP_K = 50
img_a = denorm(cat0)
img_b = denorm(cat1)
H = img_a.shape[0]                         # 128 px
patch = H // POOL                          # 16 px per cell

# Top-K (i, j) pairs by |R|.
flat = R_pair.abs().flatten()
_, idx = flat.topk(TOP_K)
pairs = [(int(i // POOL**2), int(i %  POOL**2)) for i in idx]
rmax  = R_pair.abs().max().item() + 1e-12

def patch_center(k):
    r, c = divmod(k, POOL)
    return (c + 0.5) * patch, (r + 0.5) * patch

fig = plt.figure(figsize=(11, 5.5))
gs  = GridSpec(1, 2, figure=fig, wspace=0.4)
ax_a = fig.add_subplot(gs[0]); ax_b = fig.add_subplot(gs[1])
ax_a.imshow(img_a); ax_a.axis('off'); ax_a.set_title('Cat A')
ax_b.imshow(img_b); ax_b.axis('off'); ax_b.set_title('Cat B')

# Draw bipartite lines via fig coords (axes are separate).
fig.canvas.draw()
trans_a = ax_a.transData.transform
trans_b = ax_b.transData.transform
inv     = fig.transFigure.inverted().transform
for i_a, i_b in pairs:
    val = R_pair[i_a, i_b].item()
    color = '#d73027' if val > 0 else '#4575b4'
    p_a = inv(trans_a(patch_center(i_a)))
    p_b = inv(trans_b(patch_center(i_b)))
    line = mlines.Line2D([p_a[0], p_b[0]], [p_a[1], p_b[1]],
                          transform=fig.transFigure,
                          color=color, alpha=0.55,
                          linewidth=4.0 * abs(val) / rmax)
    fig.add_artist(line)

fig.suptitle(f'VGG-16 BiLRP — top {TOP_K} patch pairs '
             f'(<phi(a),phi(b)> = {dot:.1f})', y=1.02)
plt.show()